# OCSF-RainNet — Kaggle-only 16-run continuation
Runs all remaining OCSF RainNet variants/seeds in the remote Kaggle GPU without depending on a Windows-local dispatcher. Separate Kaggle jobs cover plain seed42 and fusion seed42. This kernel writes all per-run artifacts and a master completion manifest.


In [ ]:
import json,datetime,traceback,gc,os
from pathlib import Path
_source='"""One reproducible training run for OCSF-RainNet (Kaggle GPU only).\nSix task-matched models x three fixed seeds, one run per notebook.\nSource input: private, calibrated Himawari-9 AHI B13 + IMERG V07B labels.\nNever train from the old map-rendered JPEG pilot; never evaluate test during fitting.\nOutputs are written to /kaggle/working for validated desktop artifact sync.\n"""\nimport os\nos.environ.setdefault("TF_USE_LEGACY_KERAS","1")\nos.environ["TF_CPP_MIN_LOG_LEVEL"]="2"\nos.environ["PYTHONHASHSEED"]=str(RUN_SEED)\nfrom pathlib import Path\nfrom io import BytesIO\nfrom zipfile import ZipFile\nimport numpy as np\nimport tensorflow as tf\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport hashlib, json, csv, datetime, random, math, time, traceback, sys, gc\n\nassert RUN_VARIANT in ("plain","fusion","ordinal","proposed","unet","raunet")\nassert RUN_SEED in (42,123,2025)\nnp.random.seed(RUN_SEED); random.seed(RUN_SEED); tf.random.set_seed(RUN_SEED)\ntf.keras.backend.clear_session()\nOUT=Path("/kaggle/working")\nOUT.mkdir(exist_ok=True)\nJOB=f"{RUN_VARIANT}_s{RUN_SEED}"\nFILE_PREFIX=f"OCSF_{JOB}"\nTIME=lambda:datetime.datetime.now(datetime.timezone.utc).isoformat()\nGPUS=tf.config.list_physical_devices("GPU")\nassert GPUS,"Kaggle GPU is required: no silent CPU fallback"\ntry:\n    tf.config.set_visible_devices(GPUS[:1],"GPU")\n    tf.config.experimental.set_memory_growth(GPUS[0],True)\nexcept RuntimeError:\n    pass\n\nINPUT_ROOT=Path("/kaggle/input")\nzips=list(INPUT_ROOT.rglob("paired_127_lossless.zip"))\nZIP=zips[0] if zips else None\nexploded=next((p for p in INPUT_ROOT.rglob("paired_127_lossless") if p.is_dir() and len(list(p.glob("*.npz")))==127),None)\nif ZIP is None and exploded is None:\n    raise RuntimeError("Private calibrated dataset absent: zip and Kaggle-unzipped scenes not found; do not use pilot JPEG")\n\nsource=next(iter(INPUT_ROOT.rglob("source_checksum_manifest.json")),None)\nsplits=next(iter(INPUT_ROOT.rglob("split_by_UTC_date.csv")),None)\nmodel_script=next(iter(INPUT_ROOT.rglob("rainnet_intensity_models.py")),None)\nassert source is not None and splits is not None and model_script is not None\nprovenance=json.loads(source.read_text(encoding="utf8"))\nassert len(provenance["images"])==127\nif ZIP is not None:\n    assert hashlib.sha256(ZIP.read_bytes()).hexdigest()==provenance["zip_sha256"]\n\nsys.path.insert(0,str(model_script.parent))\nimport rainnet_intensity_models as net\nnet.CHECKPOINT=next(iter(INPUT_ROOT.rglob("rainnet_binary_best_05.h5")))\nnet.ARCH=model_script.parent\nassert net.pretrained_sha256()==net.ORIGINAL_CHECKPOINT_SHA256\ndate_group={}\nwith splits.open(newline="",encoding="utf-8-sig") as f:\n    for rec in csv.DictReader(f):\n        date_group[rec["utc_date"]]=rec["assigned_split"]\nassert len(date_group)==32 and set(date_group.values())=={"train","validation","test"}\nassert [sum(1 for x in date_group.values() if x==s) for s in ("train","validation","test")]==[22,5,5]\ndataset={}\nif ZIP is not None:\n    z=ZipFile(ZIP,"r")\n    assert len([n for n in z.namelist() if n.endswith(".npz")])==127\nelse:\n    z=None\n    assert len(list(exploded.glob("*.npz")))==127\ntry:\n    for record in provenance["images"]:\n        raw=(z.read(record["scene"]) if z is not None else (exploded/record["scene"]).read_bytes())\n        assert hashlib.sha256(raw).hexdigest()==record["portable_npz_sha256"]\n        with np.load(BytesIO(raw),allow_pickle=False) as d:\n            X=np.asarray(d["x_grayscale_float32"],np.float32)\n            Y=np.asarray(d["y"],np.uint8)\n        assert X.shape==Y.shape==(185,460) and np.isfinite(X).all()\n        assert 0<=float(X.min()) and float(X.max())<=1\n        labels=np.bincount(Y[Y!=255],minlength=5)\n        assert labels.tolist()==record["classes"],(record["key"],labels.tolist(),record["classes"])\n        dataset[record["key"]]={"x":X,"y":Y,"date":record["date_utc"],"split":date_group[record["date_utc"]]}\nfinally:\n    if z is not None:z.close()\nassert len(dataset)==127\nGROUPS={s:sorted(k for k,v in dataset.items() if v["split"]==s) for s in ("train","validation","test")}\nassert [len(GROUPS[s]) for s in ("train","validation","test")]==[88,19,20]\ntrain_labels=np.zeros(5,dtype=np.int64)\nfor key in GROUPS["train"]:\n    y=dataset[key]["y"]; train_labels+=np.bincount(y[y!=255],minlength=5)\nweights=net.empirical_train_class_weights(train_labels)\nloss_fn=net.masked_weighted_ordinal_loss(weights,0.12)\nEXPECTED_PARAMS={"plain":142637,"fusion":148234,"ordinal":142632,"proposed":148217,"unet":485853,"raunet":508249}\nRUN_CFG={\n    "run_id":JOB,"variant":RUN_VARIANT,"seed":RUN_SEED,\n    "status":"TRAINING_NOT_YET_EVALUATED","start_utc":TIME(),\n    "source":"NOAA/JMA calibrated Himawari-9 AHI B13 matched to GPM IMERG Final V07B",\n    "data_identifier":"agusperdana/ocsf-rainnet-calibrated-2024",\n    "dataset_archive_sha256":provenance["zip_sha256"],\n    "pretrained_rainnet_sha256":net.ORIGINAL_CHECKPOINT_SHA256,\n    "input":"float32 grayscale derived from calibrated Kelvin, three identical channels",\n    "class_labels":"C0 [0,1), C1 [1,5), C2 [5,10), C3 [10,20], C4 (20,infinity) mm/h; 255 ignored",\n    "split_rule":"UTC date disjoint before crop; splits frozen in dataset",\n    "train_val_test_count":{k:len(v) for k,v in GROUPS.items()},\n    "date_count":{k:len(set(dataset[x]["date"] for x in v)) for k,v in GROUPS.items()},\n    "train_class_counts":train_labels.tolist(),"train_class_weights":weights,\n    "max_epochs":25,"batch_size":2,"crops_per_train_scene_per_epoch":4,\n    "crop_shape":[128,128],"learning_rates":{"warmup":0.001,"finetune":0.0001},\n    "frozen_pretrained_epochs":5 if RUN_VARIANT not in ("unet","raunet") else 0,\n    "early_stopping_patience":4,\n    "loss":"train-frequency sqrt inverse capped 12 and normalized x masked CE +0.12 ordinal severity L1",\n    "validation":"all 19 complete val scenes padded to multiple of 16, never test",\n    "test":"20 complete independent-date scenes only after best val checkpoint",\n    "dataset_admissibility":"radiometric/timestamp QC passed, broader source-independence caveats remain",\n    "software":{"tensorflow":tf.__version__,"python":sys.version.split()[0]},\n    "gpu_devices":[g.name for g in GPUS]\n}\ndef save_json(path,obj):\n    Path(path).write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding="utf8")\nsave_json(OUT/f"{FILE_PREFIX}_run_manifest.json",RUN_CFG)\n\ndef construct():\n    if RUN_VARIANT=="unet":return net.build_unet_baseline(False)\n    if RUN_VARIANT=="raunet":return net.build_unet_baseline(True)\n    return net.build_rainnet_variant(RUN_VARIANT,pretrained=True,freeze_backbone=True)\n\nmodel=construct()\nassert model.count_params()==EXPECTED_PARAMS[RUN_VARIANT],(model.count_params(),EXPECTED_PARAMS[RUN_VARIANT])\nRUN_CFG["parameters"]=model.count_params()\nsave_json(OUT/f"{FILE_PREFIX}_run_manifest.json",RUN_CFG)\ndelayed_pool={}\ndef rgb(a):\n    return np.repeat(a[...,None],3,axis=2)\ndef full(key):\n    d=dataset[key]\n    x=rgb(d["x"]); y=d["y"]\n    xp=np.pad(x,((0,7),(0,4),(0,0)),constant_values=0)\n    yp=np.pad(y,((0,7),(0,4)),constant_values=255)\n    return xp[None].astype(np.float32),yp[None].astype(np.uint8)\ndef crop_batches(epoch):\n    rng=np.random.default_rng(RUN_SEED*100003+epoch*7919)\n    keys=GROUPS["train"].copy()\n    rng.shuffle(keys)\n    patches=[]\n    for key in keys:\n        x=dataset[key]["x"]; y=dataset[key]["y"]\n        for j in range(4):\n            if rng.random()<0.55:\n                if key not in delayed_pool:\n                    delayed_pool[key]=np.argwhere((y==3)|(y==4))\n                points=delayed_pool[key]\n            else:\n                points=np.empty((0,2),dtype=np.int16)\n            if len(points):\n                pos=points[int(rng.integers(0,len(points)))]\n                r0=int(np.clip(int(pos[0])-int(rng.integers(0,128)),0,185-128))\n                c0=int(np.clip(int(pos[1])-int(rng.integers(0,128)),0,460-128))\n            else:\n                r0=int(rng.integers(0,185-128+1))\n                c0=int(rng.integers(0,460-128+1))\n            patches.append((key,r0,c0))\n    rng.shuffle(patches)\n    assert len(patches)==352\n    for i in range(0,len(patches),2):\n        xs=[];ys=[]\n        for key,r,c in patches[i:i+2]:\n            xs.append(rgb(dataset[key]["x"][r:r+128,c:c+128]))\n            ys.append(dataset[key]["y"][r:r+128,c:c+128])\n        yield tf.convert_to_tensor(np.stack(xs),tf.float32),tf.convert_to_tensor(np.stack(ys).astype(np.int32))\n\ndef validation_loss():\n    losses=[]\n    for k in GROUPS["validation"]:\n        xx,yy=full(k)\n        pp=model(tf.convert_to_tensor(xx),training=False)\n        losses.append(float(loss_fn(tf.convert_to_tensor(yy.astype(np.int32)),pp).numpy()))\n    return float(np.mean(losses))\n\ndef set_phase(epoch):\n    warmup=RUN_VARIANT not in ("unet","raunet") and epoch<=5\n    if RUN_VARIANT not in ("unet","raunet"):\n        for layer in model.rainnet_backbone_layers:\n            layer.trainable=not warmup\n    return "warmup" if (epoch<=5) else "finetune"\n\ndef optimizer_for_phase(phase):\n    return tf.keras.optimizers.Adam(learning_rate=1e-3 if phase=="warmup" else 1e-4)\n\ndef compile_step(opt):\n    @tf.function(reduce_retracing=True)\n    def train_step(x,y):\n        with tf.GradientTape() as tape:\n            probs=model(x,training=True)\n            loss=loss_fn(y,probs)\n        var=model.trainable_variables\n        grad=tape.gradient(loss,var)\n        pairs=[(g,v) for g,v in zip(grad,var) if g is not None]\n        opt.apply_gradients(pairs)\n        return loss\n    return train_step\n\nHIST=[]\nBEST=float("inf")\nWAIT=0\nBEST_EPOCH=0\nbest_path=OUT/f"{FILE_PREFIX}_best.weights.h5"\nlast_path=OUT/f"{FILE_PREFIX}_last.weights.h5"\nhistory_csv=OUT/f"{FILE_PREFIX}_epoch_history.csv"\nlast_phase=None\ntrain_step=None\ntf.config.optimizer.set_jit(False)\nstart_time=time.perf_counter()\ntry:\n    for epoch in range(1,26):\n        phase=set_phase(epoch)\n        if phase!=last_phase:\n            optimizer=optimizer_for_phase(phase)\n            train_step=compile_step(optimizer)\n            last_phase=phase\n            if epoch==6:\n                WAIT=0\n        epoch_started=time.perf_counter()\n        train_losses=[]\n        for xb,yb in crop_batches(epoch):\n            val=float(train_step(xb,yb).numpy())\n            if not np.isfinite(val):\n                raise FloatingPointError("Nonfinite training loss "+str((epoch,val)))\n            train_losses.append(val)\n        vloss=validation_loss()\n        if not np.isfinite(vloss):raise FloatingPointError("Nonfinite validation loss")\n        model.save_weights(str(last_path))\n        improvement=vloss < BEST-1e-5\n        if improvement:\n            BEST=vloss;BEST_EPOCH=epoch;WAIT=0\n            model.save_weights(str(best_path))\n        elif epoch>5:\n            WAIT+=1\n        rec={\n            "epoch":epoch,"phase":phase,"train_loss":float(np.mean(train_losses)),\n            "val_loss":vloss,"best_val_loss":BEST,"improved":improvement,\n            "wait":WAIT,"epoch_seconds":time.perf_counter()-epoch_started\n        }\n        HIST.append(rec)\n        with history_csv.open("w",newline="",encoding="utf8") as f:\n            w=csv.DictWriter(f,fieldnames=list(HIST[0]));w.writeheader();w.writerows(HIST)\n        RUN_CFG["last_epoch"]=epoch\n        RUN_CFG["best_epoch"]=BEST_EPOCH\n        RUN_CFG["best_validation_loss"]=BEST\n        RUN_CFG["status"]="TRAINING"\n        RUN_CFG["updated_utc"]=TIME()\n        save_json(OUT/f"{FILE_PREFIX}_run_manifest.json",RUN_CFG)\n        print("EPOCH",JOB,epoch,phase,"train",round(rec["train_loss"],6),"val",round(vloss,6),"best",round(BEST,6),"wait",WAIT,flush=True)\n        if epoch>5 and WAIT>=4:\n            print("EARLY_STOP",JOB,epoch,flush=True)\n            break\n    assert BEST_EPOCH>0 and best_path.exists()\n    model.load_weights(str(best_path))\nexcept BaseException as ex:\n    RUN_CFG["status"]="ERROR_BEFORE_TEST"\n    RUN_CFG["error"]=repr(ex)\n    RUN_CFG["traceback"]=traceback.format_exc()[-4000:]\n    save_json(OUT/f"{FILE_PREFIX}_run_manifest.json",RUN_CFG)\n    raise\n\ndef cm_counts(yy,pp):\n    yy=np.asarray(yy,np.uint8)\n    pp=np.asarray(pp,np.uint8)\n    mask=(yy!=255)\n    return np.bincount((yy[mask].astype(np.int64)*5+pp[mask].astype(np.int64)),minlength=25).reshape(5,5)\ndef metric(cm):\n    tp=np.diag(cm).astype(float)\n    fn=cm.sum(axis=1).astype(float)-tp\n    fp=cm.sum(axis=0).astype(float)-tp\n    iou=np.divide(tp,tp+fn+fp,out=np.full(5,np.nan),where=(tp+fn+fp)>0)\n    dice=np.divide(2*tp,2*tp+fn+fp,out=np.full(5,np.nan),where=(2*tp+fn+fp)>0)\n    micro_accuracy=float(tp.sum()/cm.sum()) if cm.sum()>0 else None\n    sev_tp=float(cm[3:5,3:5].sum())\n    sev_fn=float(cm[3:5,:3].sum())\n    sev_fp=float(cm[:3,3:5].sum())\n    return {"iou":[float(x) if np.isfinite(x) else None for x in iou],\n            "dice":[float(x) if np.isfinite(x) else None for x in dice],\n            "macro_iou":float(np.nanmean(iou)),\n            "macro_dice":float(np.nanmean(dice)),\n            "pixel_accuracy":micro_accuracy,\n            "severe_csi":float(sev_tp/(sev_tp+sev_fn+sev_fp)) if (sev_tp+sev_fn+sev_fp)>0 else None,\n            "severe_pod":float(sev_tp/(sev_tp+sev_fn)) if (sev_tp+sev_fn)>0 else None,\n            "severe_far":float(sev_fp/(sev_tp+sev_fp)) if (sev_tp+sev_fp)>0 else None,\n            "severe_tp":int(sev_tp),"severe_fn":int(sev_fn),"severe_fp":int(sev_fp)}\n\ncombined=np.zeros((5,5),np.int64)\nper_scene=[]\npredict_zip=OUT/f"{FILE_PREFIX}_test_predictions.zip"\nwith ZipFile(predict_zip,"w") as z:\n    for key in GROUPS["test"]:\n        xx,yy=full(key)\n        probs=model(tf.convert_to_tensor(xx),training=False).numpy()[0,:185,:460]\n        assert probs.shape==(185,460,5) and np.isfinite(probs).all()\n        pred=probs.argmax(-1).astype(np.uint8)\n        real=yy[0,:185,:460].copy()\n        count=cm_counts(real,pred)\n        combined+=count\n        result={"key":key,"date":dataset[key]["date"],"classes_actual":np.bincount(real[real!=255],minlength=5).tolist(),"confusion_matrix":count.tolist(),**metric(count)}\n        per_scene.append(result)\n        b=BytesIO()\n        np.savez_compressed(b,actual_C0_C4_uint8=real,predicted_C0_C4_uint8=pred,probabilities_float16=probs.astype(np.float16))\n        z.writestr(key+"_test_prediction.npz",b.getvalue())\nassert len(per_scene)==20\nwith (OUT/f"{FILE_PREFIX}_test_confusion_matrix.csv").open("w",newline="",encoding="utf8") as f:\n    wr=csv.writer(f);wr.writerow(["actual/predicted","C0","C1","C2","C3","C4"])\n    for i,row in enumerate(combined):wr.writerow([f"C{i}",*row])\nwith (OUT/f"{FILE_PREFIX}_test_per_scene_metrics.csv").open("w",newline="",encoding="utf8") as f:\n    keys=["key","date","macro_iou","macro_dice","pixel_accuracy","severe_csi","severe_pod","severe_far","severe_tp","severe_fn","severe_fp"]\n    wr=csv.DictWriter(f,fieldnames=keys);wr.writeheader()\n    for sc in per_scene:wr.writerow({k:sc.get(k) for k in keys})\nresults={"project":"OCSF-RainNet","run":JOB,"model":RUN_VARIANT,"seed":RUN_SEED,"status":"COMPLETE",\n         "data_source":"Calibrated AHI B13 / satellite IMERG V07B, NOT independent BMKG observations",\n         "train_scene_count":88,"validation_scene_count":19,"test_scene_count":20,\n         "test_UTC_dates":sorted(set(dataset[k]["date"] for k in GROUPS["test"])),\n         "epoch_history":HIST,"epochs_executed":len(HIST),"best_epoch":BEST_EPOCH,\n         "best_validation_loss":BEST,"parameters":model.count_params(),\n         "total_training_seconds":time.perf_counter()-start_time,\n         "aggregate_test_confusion_matrix":combined.tolist(),\n         "aggregate_test_metrics":metric(combined),\n         "per_scene":per_scene,"finished_utc":TIME(),\n         "rainnet_checkpoint_sha256":net.ORIGINAL_CHECKPOINT_SHA256,\n         "data_archive_sha256":provenance["zip_sha256"]}\nsave_json(OUT/f"{FILE_PREFIX}_test_results.json",results)\nfig,ax=plt.subplots(figsize=(7,6))\nim=ax.imshow(np.log1p(combined),cmap="Blues")\nax.set_xticks(range(5),[f"C{i}" for i in range(5)])\nax.set_yticks(range(5),[f"C{i}" for i in range(5)])\nax.set_xlabel("Predicted rainfall class")\nax.set_ylabel("Actual IMERG rainfall class")\nax.set_title("OCSF-RainNet held-out confusion matrix: "+JOB)\nfor i in range(5):\n    for j in range(5):\n        ax.text(j,i,f"{combined[i,j]:,}",ha="center",va="center",fontsize=8,color="white" if np.log1p(combined[i,j])>0.6*np.log1p(combined.max()) else "black")\nfig.colorbar(im,ax=ax,label="log(1 + cell count)")\nfig.tight_layout()\nfig.savefig(OUT/f"{FILE_PREFIX}_confusion_600dpi.png",dpi=600,bbox_inches="tight")\nplt.close(fig)\nfig,ax=plt.subplots(figsize=(7,4))\nax.plot([h["epoch"] for h in HIST],[h["train_loss"] for h in HIST],label="Train (crop)")\nax.plot([h["epoch"] for h in HIST],[h["val_loss"] for h in HIST],label="Validation (full scene)")\nax.axvline(BEST_EPOCH,color="gray",linestyle="--",label="Best epoch")\nax.set_xlabel("Epoch");ax.set_ylabel("Masked weighted loss")\nax.set_title("OCSF-RainNet learning curves: "+JOB);ax.legend()\nfig.tight_layout()\nfig.savefig(OUT/f"{FILE_PREFIX}_learning_curves_600dpi.png",dpi=600,bbox_inches="tight")\nplt.close(fig)\nRUN_CFG.update({"status":"COMPLETE","finished_utc":TIME(),"best_epoch":BEST_EPOCH,\n    "test_macro_iou":results["aggregate_test_metrics"]["macro_iou"],\n    "test_macro_dice":results["aggregate_test_metrics"]["macro_dice"],\n    "test_severe_csi":results["aggregate_test_metrics"]["severe_csi"]})\nsave_json(OUT/f"{FILE_PREFIX}_run_manifest.json",RUN_CFG)\n(OUT/f"{FILE_PREFIX}_RUN_COMPLETE.flag").write_text(TIME(),encoding="utf8")\nprint("RUN_COMPLETE",JOB,results["aggregate_test_metrics"],"epochs",len(HIST),flush=True)\n'
_jobs=[('ordinal', 42), ('proposed', 42), ('unet', 42), ('raunet', 42), ('plain', 123), ('fusion', 123), ('ordinal', 123), ('proposed', 123), ('unet', 123), ('raunet', 123), ('plain', 2025), ('fusion', 2025), ('ordinal', 2025), ('proposed', 2025), ('unet', 2025), ('raunet', 2025)]
_out=Path('/kaggle/working')
_log={'mode':'KAGGLE_REMOTE_16_JOB_MASTER','created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'jobs':[]}
for variant,seed in _jobs:
    print('MASTER_START',variant,seed,flush=True)
    _status={'variant':variant,'seed':seed,'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
    try:
        _scope={'RUN_VARIANT':variant,'RUN_SEED':seed,'__name__':'__main__'}
        exec(compile(_source,'ocsf_kaggle_train_one.py','exec'),_scope)
        _status['status']='COMPLETE'
    except BaseException as error:
        _status['status']='ERROR'
        _status['error']=repr(error)
        _status['traceback']=traceback.format_exc()[-5000:]
        print('MASTER_FAILED',variant,seed,repr(error),flush=True)
    finally:
        _status['finished_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat()
        _log['jobs'].append(_status)
        (_out/'OCSF_16_Run_Master_Status.json').write_text(json.dumps(_log,indent=2),encoding='utf8')
        gc.collect()
_log['completed_count']=sum(x['status']=='COMPLETE' for x in _log['jobs'])
_log['finished_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat()
(_out/'OCSF_16_Run_Master_Status.json').write_text(json.dumps(_log,indent=2),encoding='utf8')
print('MASTER_COMPLETED',_log['completed_count'],'OF',len(_jobs),flush=True)
assert _log['completed_count']==len(_jobs),'At least one job failed; inspect OCSF_16_Run_Master_Status.json'
